In [3]:
print("IntelliDocs AI")

IntelliDocs AI


In [4]:
from langchain_ollama import OllamaLLM
llm=OllamaLLM(model="gemma:2b")
response=llm.invoke("What is generative Ai? Explain in one sentence.")
print(response)

Sure. Here's a one-sentence explanation of generative AI:

Generative AI is a type of artificial intelligence that can create new, realistic content, such as images, text, music, and code, based on existing data.


In [6]:
from langchain_ollama import OllamaEmbeddings
embeddings=OllamaEmbeddings(model="qwen3-embedding:0.6b")
vetor=embeddings.embed_query("what is generative ai?")
print(len(vetor))
print(vetor[:5])

1024
[-0.022292411, -0.024912657, -0.007768587, -0.08131329, 0.026428651]


In [7]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('../data/My.pdf')
docs=loader.load()
docs

C:\Users\pavan\AppData\Local\Temp\ipykernel_10636\2873226728.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'WeasyPrint 65.1', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Audio and Video Embedding - Web Development Course', 'source': '../data/My.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Audio and Video Embedding in HTML\nEmbedding Audio\nUse the <audio>  tag to add sound or music to your webpage.\nBasic Example:\ncontrols  adds play, pause, and volume controls.\nThe <source>  tag specifies the audio file and type.\nAudio Formats\nFormat MIME Type\nMP3 audio/mpeg\nOGG audio/ogg\nWAV audio/wav\nTo support all browsers, you can include multiple sources:\n<audio controls>\n<source src="audio.mp3" type="audio/mpeg">\n  Your browser does not support the audio element.\n</audio>\n• \n• \n<audio controls>\n<source src="audio.mp3" type="audio/mpeg">'),
 Document(metadata={'producer': 'WeasyPrint 65.1', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Audio and Video Embedding - Web Development Course', 'source': '../data/My.pdf', 'tota

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunk=text_splitter.split_documents(docs)
print("Number of chunks:",len(chunk))
print(chunk[0])

Number of chunks: 5
page_content='Audio and Video Embedding in HTML
Embedding Audio
Use the <audio>  tag to add sound or music to your webpage.
Basic Example:
controls  adds play, pause, and volume controls.
The <source>  tag specifies the audio file and type.
Audio Formats
Format MIME Type
MP3 audio/mpeg
OGG audio/ogg
WAV audio/wav
To support all browsers, you can include multiple sources:
<audio controls>
<source src="audio.mp3" type="audio/mpeg">
  Your browser does not support the audio element.
</audio>
• 
•' metadata={'producer': 'WeasyPrint 65.1', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Audio and Video Embedding - Web Development Course', 'source': '../data/My.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


In [9]:
from langchain_ollama import OllamaEmbeddings
embeddings=OllamaEmbeddings(model="qwen3-embedding:0.6b")
vector=embeddings.embed_query(chunk[0].page_content)
print("vector lenght:",len(vector))
print(vector)

vector lenght: 1024
[-0.0277592, -0.049602784, -0.0016088877, -0.095265225, 0.049569808, 0.010239387, 0.010990509, 0.057494063, -0.033118416, -0.008592228, -0.030939305, 0.008702602, -0.0037233247, -0.0020678027, -0.009245394, 0.0592467, -0.021458674, 0.06775634, 0.0053593135, -0.020822462, -0.043390263, 0.04232975, -0.06971231, 0.06237477, -0.017473646, 0.01679949, 0.042397175, -0.033863682, -0.03742859, 0.0031914895, -0.05302444, 0.0032997753, 0.009351455, 0.025098497, -0.036071, -0.004565135, 0.06505995, 0.03194281, 0.00771075, 0.021688525, -0.0059697027, -0.0058957096, 0.017730018, 0.018386263, -0.006406259, -0.02584943, 0.0041977693, 0.038643204, -0.004101021, -0.0017614783, -0.018535543, 0.0048470977, -0.012157781, -0.026854036, -0.021905692, 0.01059908, 0.046236362, 0.08571378, 0.002863461, 0.0066986405, -0.03746114, 0.028333262, 0.01753218, 0.082115754, 0.027144175, -0.022461621, -0.02147206, -0.0010422509, -0.031102384, 0.0015639685, 0.021866972, 0.020208474, -0.0076486846, 0.

In [10]:
##ChromaDB stores the chunks and their vector representations,so later ehrn you ask question ,we search for the most relevent chunks


from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(
    documents=chunk,
    embedding=embeddings,
    collection_name="intellidocs"
)
print("Documents stored:",vectorstore._collection.count())

Documents stored: 5


In [11]:
##Test Retrieval-->RAG needs to find the relevant chunk when you ask a question
query="What is this document about?"
results=vectorstore.similarity_search(query,k=3)
for i,doc in enumerate(results):
    print(f".\n-----Result{i+1}---")
    print(doc.page_content[:500])

.
-----Result1---
</audio>
• 
• 
<audio controls>
<source src="audio.mp3" type="audio/mpeg">
.
-----Result2---
</video>
• 
• 
<video controls>
<source src="movie.mp4" type="video/mp4">
<source src="movie.ogg" type="video/ogg">
</video>
.
-----Result3---
Tip
Always provide controls  so users can interact with media.
Use multiple formats for broader browser support.
Include fallback text for unsupported browsers.
• 
• 
•


In [15]:
from langchain_ollama import OllamaLLM
llm=OllamaLLM(model="gemma:2b")
query = "What should you provide so users can interact with media?"
results=vectorstore.similarity_search(query,k=3)
context="\n\n".join([doc.page_content for doc in results])

prompt=f"""
Answer the question using the context below.
Contex:
{context}

Question:
{query}

Answer:
"""
response=llm.invoke(prompt)
print(response)

The context provides that users should provide controls so they can interact with the media.


Note:RAG allows an LLM to answer your question using information retrieved from your own documents.
"RAG retrieves relevant information from a knowledge source and provides it as context to an LLM so the LLM can generate a grounded answer."

In [16]:
##Step 2: Add source/page references 📄

##Right now your chatbot gives an answer, but the user doesn't know where the answer came from.

for i,doc in enumerate(results):
    print(f"Source{i+1}:")
    print("File:",doc.metadata.get("source"))
    print("Page:",doc.metadata.get("page"))
    print()

Source1:
File: ../data/My.pdf
Page: 2

Source2:
File: ../data/My.pdf
Page: 1

Source3:
File: ../data/My.pdf
Page: 0



In [18]:
query = "What should you provide so users can interact with media?"

results = vectorstore.similarity_search(query, k=3)

context = "\n\n".join([doc.page_content for doc in results])

prompt = f"""
Answer the question using only the context below.
If the answer is present, answer directly and briefly.

Context:
{context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print("ANSWER:")
print(response)

print("\nSOURCES:")
for i, doc in enumerate(results):
    print(f"{i+1}. {doc.metadata.get('source')} - Page {doc.metadata.get('page')}")

ANSWER:
The context says that the answer is "Controls". 

Therefore, the answer is: Controls

SOURCES:
1. ../data/My.pdf - Page 2
2. ../data/My.pdf - Page 1
3. ../data/My.pdf - Page 0


In [ ]:
##Find every PDF inside the data folder and load all of them.
import os  ##os is a Python module that lets us work with files and folders.
from langchain_community.document_loaders import PyPDFLoader
data_folder="../data"
all_docs=[]   ##We're going to put all the pages from all PDFs into this list.
for file in os.listdir(data_folder):
    if file.endswith(".pdf"):
        loader=PyPDFLoader(os.path.join(data_folder,file))   ##os.path.join() this combines ../data/My.pdf
        docs=loader.load()
        all_docs.extend(docs)

print("Total pages loaded:",len(all_docs))

Total pages loaded: 4


In [ ]:
##Chunk all the PDFs
##So all_chunks will contain chunks from both PDFs.
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

all_chunks = text_splitter.split_documents(all_docs)

print("Total chunks:", len(all_chunks))

Total chunks: 7


In [22]:
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="intellidocs_multi"
)

print("Documents stored:", vectorstore._collection.count())

Documents stored: 7


In [23]:
## Testing whether it reads from both pdf
query = "What is CSS used for?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))


--- Result 1 ---
button, and button.
4. CSS Selectors
CSS selectors identify which HTML elements should receive styles. Common selectors include element
selectors, class selectors, and ID selectors.
5. HTML vs CSS
HTML defines the structure and content of a webpage, while CSS controls its presentation and visual
appearance.
Source: ../data\Web_Development_Notes.pdf
Page: 0

--- Result 2 ---
Web Development Notes
1. HTML
HTML (HyperText Markup Language) is used to create the structure of web pages. Common elements
include headings, paragraphs, links, images, lists, tables, audio, and video.
2. CSS
CSS (Cascading Style Sheets) is used to control the appearance of HTML elements. It can change colors,
fonts, spacing, borders, layouts, and responsive behavior.
3. HTML Forms
HTML forms collect user input. Common form elements include input, label, textarea, select, checkbox, radio
Source: ../data\Web_Development_Notes.pdf
Page: 0

--- Result 3 ---
Tip
Always provide controls  so users can i

In [26]:
def ask_question(query):

    results = vectorstore.similarity_search(query, k=3)

    context = "\n\n".join([doc.page_content for doc in results])

    prompt = f"""
    Answer the question using only the context below.
    If the answer is not present in the context, say you don't know.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = llm.invoke(prompt)

    print("ANSWER:")
    print(response)

    print("\nSOURCES:")

    sources = set()

    for doc in results:
        source = doc.metadata.get("source")
        page = doc.metadata.get("page")
        sources.add((source, page))

    for source, page in sources:
        print(f"- {source} - Page {page}")

In [27]:
ask_question("What is CSS used for?")


ANSWER:
According to the context, CSS is used to control the appearance of HTML elements. It can change colors, fonts, spacing, borders, layouts, and responsive behavior.

SOURCES:
- ../data\Web_Development_Notes.pdf - Page 0
- ../data\My.pdf - Page 2


In [28]:
ask_question("What should you provide so users can interact with media?")

ANSWER:
The context says that users should provide controls so they can interact with media.

SOURCES:
- ../data\My.pdf - Page 0
- ../data\My.pdf - Page 1
- ../data\My.pdf - Page 2
